[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/dirac/dirac.ipynb)

# The Dirac Electron in the Spacetime Algebra

This example computes the spacetime frame, charge current, intrinsic spin, plane-wave energies, and trembling trajectory of a relativistic Dirac electron. A free electron with pure positive energy drifts steadily at its group velocity, whereas mixing positive and negative energy makes its charge circulate near the speed of light around a loop smaller than its Compton wavelength, the motion Schrödinger identified in 1930 and named Zitterbewegung. In natural units with the reduced Planck constant, the speed of light, and the electron rest mass all equal to one, lengths are measured in reduced Compton wavelengths, 386 femtometres, and times in light-crossing times across one reduced Compton wavelength.

The electron's state at an event is modelled as an even multivector of the spacetime algebra built from a scalar density, a chiral angle beta, and a Lorentz rotor. Sandwiching vectors with the spinor (`psi >> Vector`) carries the observer's frame to the electron's own scaled frame: `psi >> time` gives the charge current, and `(psi >> spin_plane).dual()` gives the intrinsic turning-plane patch of the spin. Composing `psi >> Bivector` with the inverse outermorphism of that vector frame isolates `(pseudoscalar * beta).exp() / density`, while right multiplication by the rest-frame spin plane (`Spinor * spin_plane`) squares to minus one and commutes with the plane-wave Dirac Hamiltonian, following Hestenes ([*Space-Time Algebra*](https://doi.org/10.1007/978-3-319-18413-5)).

In complex four-component Dirac matrix notation, the state is written as a column bispinor $\psi \in \mathbb{C}^4$ acted on by $4 \times 4$ gamma matrices $\gamma^\mu$ with Dirac adjoint $\bar\psi = \psi^\dagger \gamma^0$, reading out the conserved four-current $J^\mu = \bar\psi \gamma^\mu \psi$, the axial spin pseudovector $\bar\psi \gamma^\mu \gamma^5 \psi$ with $\gamma^5 = i\gamma^0\gamma^1\gamma^2\gamma^3$, and the scalar and pseudoscalar invariants $\bar\psi \psi$ and $i\bar\psi \gamma^5 \psi$ component by component. Plane-wave evolution $i\partial_t \psi = (\boldsymbol{\alpha}\cdot\mathbf{p} + \beta m)\psi$ couples upper and lower two-spinor blocks through $4 \times 4$ matrices $\alpha^k = \gamma^0 \gamma^k$ and $\beta = \gamma^0$, and Zitterbewegung arises from cross terms oscillating at $e^{\pm 2iEt}$ in the Heisenberg-picture velocity operator $\boldsymbol{\alpha}(t)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA as ga
from examples.relativity.dirac import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=3, suppress=True)

# Spacetime algebra with spatial axes positive and time negative, `x+y+z+t-`.
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Vector = ga.gatype.vector()                         # Vector+
Bivector = ga.gatype.bivector()                     # Bivector+
Spinor = ga.gatype.even()                           # Spinor+
Antivector = ga.gatype.antivector()                 # Antivector-
Spin = ga.gatype.antibivector()                     # Spin-
Spatial = Vector.derive.within(Algebra("x+y+z+"))   # Vector+
Frame = ga.gatype((Vector, Vector))                 # Vector+ <- Vector+
Duality = ga.gatype((Bivector, Bivector))           # Bivector+ <- Bivector+
Hamiltonian = ga.gatype((Spinor, Spinor))           # Spinor+ <- Spinor+

# Observer time direction pairing to +1 with `mv.t`, unit spacetime volume, and rest-frame spin plane.
time = -mv.t.dual()                                 # [] Antivector-
pseudoscalar = mv.xyzt                              # [] Pseudoscalar+
spin_plane = mv.xy                                  # [] Bivector+
phase_unit = Spinor * spin_plane                    # [] Spinor+ <- Spinor+

## Spinor action on spacetime

An even spacetime multivector has eight real components built from a density, an angle beta, and a six-parameter Lorentz rotor. Sandwiching spacetime vectors with the spinor (`psi >> Vector`) applies the rotor and scales by the density, carrying the observer's frame to the electron's own: `psi >> time` gives the conserved charge current, and `(psi >> spin_plane).dual()` gives the intrinsic turning-plane patch of the spin. Meeting that spin patch with the observer's slice cuts out the spatial spin axis, while `psi.symmetric_reverse_product()` recovers the density split between a scalar and a pseudoscalar at the angle beta.

In Hestenes' spacetime-algebra notation, the canonical decomposition reads $\psi = \sqrt{\rho}\,e^{I\beta/2}R$, with $\psi\tilde{\psi} = \rho\,e^{I\beta}$.

In [ ]:
# A boost along +z combined with a turn in the xy plane, together with a density and angle beta.
rotor = ((mv.tz * -0.6 + mv.xy * 0.8) * 0.5).exp()     # [] Spinor+
density, beta = 1.5, 0.4

In [ ]:
def dirac_spinor(density: float, beta: float, rotor: Spinor) -> Spinor:
    """A density, a chiral mixing angle `beta`, and a Lorentz `rotor` combine into an even spacetime
    spinor (`[] Spinor+`)."""
    chiral_factor = (pseudoscalar * (beta / 2)).exp()   # [] Spinor+
    return chiral_factor * rotor * np.sqrt(density)     # [] Spinor+


def current_velocity(psi: Spinor) -> Antivector:
    """Dividing the electron's spacetime current by the charge density across the observer's slice
    and subtracting the observer's time direction leaves the spatial drift velocity (`[...] Antivector-`)."""
    flow = psi >> time                                  # [...] Antivector-
    charge_density = mv.t & flow                        # [...] Scalar+
    return flow / charge_density - time                 # [...] Antivector-


psi = dirac_spinor(density, beta, rotor)                # [] Spinor+
frame = psi >> Vector                                   # [] Vector+ <- Vector+
current = psi >> time                                   # [] Antivector-
spin = (psi >> spin_plane).dual()                       # [] Spin-
invariants = psi.symmetric_reverse_product()            # [] Spinor+

In [ ]:
print("charge density across slice: ", (mv.t & current).to_array())
print("drift velocity:\n", current_velocity(psi).coefficient_table())
print("spin patch:\n", spin.coefficient_table())
print("current Lorentz norm:        ", current.norm().to_array())
print("rho cos(beta), rho sin(beta):", invariants.cast(Scalar).to_array(), -(invariants * pseudoscalar).cast(Scalar).to_array())

## Bivector duality and the angle beta

Because the pseudoscalar anticommutes with every vector, the chiral factor `(pseudoscalar * (beta / 2)).exp()` cancels out of `psi >> Vector` and leaves the current untouched. On bivectors the pseudoscalar commutes instead: the frame's outermorphism `frame.outermorphism(Bivector)` rotates a spacetime plane by the Lorentz rotor and scales by the squared density, whereas `psi >> Bivector` also multiplies by `(pseudoscalar * beta).exp()` and mixes the spin plane partly into its complement. Composing `psi >> Bivector` with the inverse outermorphism isolates `(pseudoscalar * beta).exp() / density` directly.

In complex Dirac matrix notation, beta is called the Yvon-Takabayasi angle, extracted as the complex argument of $\bar\psi\psi + i\bar\psi\gamma^5\psi$.

In [ ]:
def bivector_duality(psi: Spinor) -> Duality:
    """Composing the spinor's bivector sandwich with the inverse outermorphism of its vector frame
    isolates multiplication by `(pseudoscalar * beta).exp() / density` (`[...] Bivector+ <- Bivector+`)."""
    frame = psi >> Vector                               # [...] Vector+ <- Vector+
    extended = frame.outermorphism(Bivector)            # [...] Bivector+ <- Bivector+
    own = psi >> Bivector                               # [...] Bivector+ <- Bivector+
    return own(extended.inverse())                      # [...] Bivector+ <- Bivector+


def spin_axis(psi: Spinor) -> Antivector:
    """Meeting the intrinsic spin patch `(psi >> spin_plane).dual()` with the observer's slice cuts
    out the spatial spin axis (`[...] Antivector-`)."""
    spin_patch = (psi >> spin_plane).dual()             # [...] Spin-
    return spin_patch ^ mv.t                            # [...] Antivector-


duality_map = bivector_duality(psi)                     # [] Bivector+ <- Bivector+
duality_factor = duality_map(spin_plane) * spin_plane.inverse()   # [] Spinor+

In [ ]:
print("duality scalar and pseudoscalar:", duality_factor.cast(Scalar).to_array(), -(duality_factor * pseudoscalar).cast(Scalar).to_array())
print("cos(beta) / rho, sin(beta) / rho:", np.cos(beta) / density, np.sin(beta) / density)

## Plane-wave energies

For a plane wave of fixed spatial momentum, the Dirac equation reduces to a symmetric linear Hamiltonian on spinors: spatial momentum multiplies from the left as a relative spacetime bivector, and rest mass acts through reflection across the observer's slice. Right multiplication by the rest-frame spin plane (`Spinor * spin_plane`) squares to minus one and commutes with this Hamiltonian, pairing each spin state with its quarter-turn phase partner so the eigenvalues are minus and plus the relativistic energy, each fourfold degenerate. Positive-energy eigenstates have `beta == 0` with positive scalar reverse norm, whereas negative-energy eigenstates have `beta == np.pi` with negative scalar reverse norm.

In complex Dirac matrix notation, the plane-wave equation $(i\gamma^\mu p_\mu - m)\psi = 0$ is rearranged into the $4 \times 4$ Hermitian eigenproblem $H\psi = (\boldsymbol{\alpha}\cdot\mathbf{p} + \beta m)\psi = E\psi$ with energies $\pm\sqrt{\mathbf{p}^2 + m^2}$, where right multiplication by the spin plane corresponds to scalar multiplication by the complex unit $i$.

In [ ]:
mass = scenarios.mass
sample_momentum = mv(Spatial, [0.3, 0.0, 0.4])          # [] Vector+

In [ ]:
def plane_wave_hamiltonian(momentum: Vector, mass: float) -> Hamiltonian:
    """Left multiplication by the relative momentum and slice reflection scaled by `mass` form the
    plane-wave Dirac Hamiltonian on spinors (`[...] Spinor+ <- Spinor+`)."""
    return (momentum * mv.t) * Spinor + mass * (mv.t * Spinor * mv.t.inverse())


def plane_wave_energy(momentum: Vector, mass: float) -> Scalar:
    """Mass and spatial momentum determine the positive relativistic energy of a free plane wave
    (`[...] Scalar+`)."""
    return (mass**2 + (momentum | momentum)).square_root()


eigenvalues, eigenstates = plane_wave_hamiltonian(sample_momentum, mass).eigh()   # [8] Scalar+, [8] Spinor+
state_signs = eigenstates.scalar_norm_squared()                                   # [8] Scalar+
print("energies:     ", eigenvalues.to_array())
print("rho cos(beta):", state_signs.to_array())

## Zitterbewegung

Superposing a positive-energy eigenstate and a negative-energy eigenstate of the same spatial momentum makes the two components counter-rotate in `spin_plane` at the plane-wave energy. Where they interfere, the charge current swings around in the plane perpendicular to the spin at twice the energy, tracing a helix of radius up to half a reduced Compton wavelength. Because the negative-energy component drifts opposite to the momentum, increasing its share slows the upward drift along the momentum axis until an equal mixture orbits in a closed horizontal loop.

In Heisenberg-picture Dirac matrix mechanics, integrating $\dot{\mathbf{x}}(t) = \boldsymbol{\alpha}(t) = \mathbf{p}H^{-1} + (\boldsymbol{\alpha}(0) - \mathbf{p}H^{-1})e^{-2iHt}$ separates this steady drift $\mathbf{p}H^{-1}$ from the $2E$ oscillatory cross term between positive-energy and negative-energy projections.

In [ ]:
drift_momentum = mv(Spatial, scenarios.drift_coordinates)   # [] Vector+
mixtures = scenarios.mixtures
dt = scenarios.dt
n_frames, n_substeps = scenarios.n_frames, scenarios.n_substeps
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def mixed_spinors(momentum: Vector, mass: float, mixtures: np.ndarray) -> Spinor:
    """Superposing unit-density positive-energy and negative-energy eigenstates of `momentum`
    produces initial spinors (`[n_mixtures] Spinor+`) with negative-energy weights `mixtures`."""
    hamiltonian = plane_wave_hamiltonian(momentum, mass)                        # [] Spinor+ <- Spinor+
    energy = plane_wave_energy(momentum, mass)                                  # [] Scalar+
    identity = mv.rotor()                                                       # [] Spinor+
    # Project unit-spinor seeds onto positive and negative energy, turning in opposite senses.
    electron = 0.5 * (identity + hamiltonian(identity) / energy)                # [] Spinor+
    positron = 0.5 * (mv.tx - hamiltonian(mv.tx) / energy)                      # [] Spinor+
    electron = electron / electron.scalar_norm_squared().square_root()          # [] Spinor+
    positron = positron / (-positron.scalar_norm_squared()).square_root()       # [] Spinor+
    return electron * np.sqrt(1 - mixtures) + positron * np.sqrt(mixtures)      # [n_mixtures] Spinor+


def trembling_states(
    momentum: Vector, mass: float, initial: Spinor, dt: float, n_frames: int, n_substeps: int
) -> Iterator[tuple[Antivector, Spinor]]:
    """Yields `(trail, spinor)` states (`[n_mixtures, ...] Antivector-`, `[n_mixtures] Spinor+`) as
    positive-energy and negative-energy spinor components counter-rotate in `spin_plane`."""
    hamiltonian = plane_wave_hamiltonian(momentum, mass)                        # [] Spinor+ <- Spinor+
    energy = plane_wave_energy(momentum, mass)                                  # [] Scalar+
    positive = 0.5 * (initial + hamiltonian(initial) / energy)                  # [n_mixtures] Spinor+
    negative = initial - positive                                               # [n_mixtures] Spinor+
    step_turn = (spin_plane * (energy * dt)).exp()                              # [] Spinor+
    position = 0 * (initial >> time)                                            # [n_mixtures] Antivector-
    positions: list[Antivector] = []
    for _ in range(n_frames):
        for _ in range(n_substeps):
            psi = positive + negative                                           # [n_mixtures] Spinor+
            positions.append(position)
            position = position + current_velocity(psi) * dt                    # [n_mixtures] Antivector-
            positive = positive * step_turn.reverse()                           # [n_mixtures] Spinor+
            negative = negative * step_turn                                     # [n_mixtures] Spinor+
        yield stack(positions, axis=-1), psi


initial_spinors = mixed_spinors(drift_momentum, mass, mixtures)                 # [3] Spinor+
streamed = show_states(
    trembling_states(drift_momentum, mass, initial_spinors, dt, n_frames, n_substeps),
    render.paths_animation,
    name="dirac_trembling",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

In [ ]:
final_paths = streamed[-1][0]                                                    # [3, 600] Antivector-
save_figure(render.paths_figure(streamed), "dirac_trembling");

The blue helix (`10%` negative energy) climbs fastest along the vertical momentum axis with narrow coils, the purple helix (`25%` negative energy) climbs at half that rate with wider coils, and the red circle (`50%` negative energy) stays in the horizontal plane with zero net drift while the terminal arrows track each electron's spatial spin axis.

In [ ]:
# checks
# The current's Lorentz norm equals the density, and bivector_duality isolates exp(I * beta) / density.
assert ((current.norm() - density).abs() < 1e-8).all()
duality_error = duality_factor - (pseudoscalar * beta).exp() / density          # [] Spinor+
assert (np.abs(duality_error.kernel) < 1e-9).all()

# Plane-wave energies are fourfold -E and +E, and right multiplication by spin_plane squares to -1.
expected_energy = plane_wave_energy(sample_momentum, mass)                      # [] Scalar+
assert ((eigenvalues[:4] + expected_energy).abs() < 1e-10).all()
assert ((eigenvalues[4:] - expected_energy).abs() < 1e-10).all()
assert (state_signs[:4] < 0).all() and (state_signs[4:] > 0).all()
probe = mv(Spinor, np.random.default_rng(0).normal(size=8))                     # [] Spinor+
assert (np.abs((phase_unit(phase_unit(probe)) + probe).kernel) < 1e-12).all()

# Charge density stays conserved across all streamed frames, and the 50% admixture has zero z-drift.
frame_spinors = stack([state_psi for _, state_psi in streamed], axis=-1)        # [3, n_frames] Spinor+
densities = mv.t & (frame_spinors >> time)                                      # [3, n_frames] Scalar+
assert ((densities - densities[:, :1]).abs() < 1e-10).all()
assert ((mv.z & final_paths[-1]).abs() < 1e-2).all()